In [ ]:
def create_landsat_df(buffer_meters, filtered_images, points, max_elements):
    print("PROCESSING LANDSAT IMAGES")

    # Extract information from each satellite image.
    data_list_info = get_data_list(filtered_images,
                                   points,
                                   max_elements,
                                   buffer_meters=30,
                                   dataset="Landsat")

    # Append data and convert to pandas dataframe
    df_l30_data = []

    for feature in data_list_info['features']:
        props = feature['properties']
        #print("")
        #print(props)

        blue  = props.get('B2') # Blue band  - ~0.45–0.51 µm
        green = props.get('B3') # Green band - ~0.53–0.59 µm
        red   = props.get('B4') # Red band   - ~0.64–0.67 µm
        nir   = props.get('B5') # NIR - Near Infrared band
        swir1 = props.get('B6') # SWIR1 - Shortwave Infrared Band 1
        swir2 = props.get('B7') # SWIR2 - Shortwave Infrared Band 2

        lon = props.get('orig_lon')
        lat = props.get('orig_lat')
        row_id = props.get('row_id')

        #ndvi  = (nir - red) / (nir + red)
        #mndwi = (green - swir1) / (green + swir1)
        #nbr   = (nir - swir2) / (nir + swir2)
        #evi_denom = nir + 6 * red - 7.5 * blue + 1
        #evi = 2.5 * ((nir - red) / evi_denom)

        ndvi  = safe_div(nir, red)
        mndwi = safe_div(green, swir1)
        nbr   = safe_div(nir, swir2)
        if None in (nir, red, blue):
            evi = None
        else:
            denom = nir + 6 * red - 7.5 * blue + 1
            evi = 2.5 * ((nir - red) / denom) if denom != 0 else None

        df_l30_data.append({
            'time'  : props.get('time'),
            'row_id': props.get('row_id'),

            'longitude': lon,
            'latitude' : lat,

            'Blue' : blue,
            'Green': green,
            'Red'  : red,

            'NIR'  : nir,
            'SWIR1': swir1,
            'SWIR2': swir2,

            'NDVI' : ndvi,
            'MNDWI': mndwi,
            'NBR'  : nbr,
            'EVI'  : evi,

            'CLOUD_COVERAGE': props.get('CLOUD_COVERAGE')
        })

    print("")
    return pd.DataFrame(df_l30_data)

In [ ]:
# Process one batch with same date range
def process_date_group_landsat_only(df_group, buffer_meters, max_elements):

    # Extract common date range
    start_date = str(df_group['start_date'].iloc[0])
    end_date   = str(df_group['end_date'].iloc[0])

    # Build points for this group
    points = build_points_from_group(df_group)
    print("Group rows:", len(df_group))
    print("Point count sent to EE:", points.size().getInfo())
    
    # Get filtered image collections
    filter_sentinel, filter_landsat = create_filters_GE(start_date, end_date, points)

    # Extract Landsat data
    df_l30 = create_landsat_df(buffer_meters, filter_landsat, points, max_elements)
    df_l30_final = reduce_df_by_median_across_images(df_l30)

    # Ensure all row_ids are present
    base_df = pd.DataFrame({'row_id': df_group['row_id'].astype(int)})
    df_l30_final = base_df.merge(df_l30_final, on='row_id', how='left')

    # Merge everything together
    merged = df_group.merge(df_l30_final, on='row_id', how='left')

    return merged